# Statistical Arbitrage: Pairs Trading Analysis
### Quantitative Research Portfolio by **Rhameyza Faiqo Susanto**

---

**Objective:** Menganalisis dan membangun strategi Pairs Trading antara dua aset yang berkorelasi tinggi (Coca-Cola/KO & PepsiCo/PEP) menggunakan pendekatan Statistical Arbitrage.

**Metodologi:**
1. Data Fetching via `yfinance`
2. Uji Kointegrasi (Engle-Granger)
3. Hedge Ratio via OLS Regression
4. Rolling Z-Score & Trading Signals
5. Interactive Visualization (Plotly)

---

## Cell 1: Install & Import Libraries
```
pip install yfinance pandas numpy plotly statsmodels
```

In [ ]:
# ============================================================
# CELL 1: IMPORT LIBRARIES
# ============================================================

import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint
import warnings

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.4f}'.format)

print("All libraries imported successfully.")
print("Pairs Trading Analysis Engine ready.")

## Cell 2: Data Fetching
Mengunduh data Adjusted Close dari Coca-Cola (KO) dan PepsiCo (PEP) selama 5 tahun terakhir.

In [ ]:
# ============================================================
# CELL 2: DATA FETCHING via yfinance
# ============================================================
# Mengunduh data historis Adjusted Close dari dua saham yang
# berkorelasi tinggi. auto_adjust=True memastikan harga sudah
# memperhitungkan stock splits dan dividen.
# ------------------------------------------------------------

TICKER_1 = 'KO'
TICKER_2 = 'PEP'
PERIOD   = '5y'

print(f"Fetching data for {TICKER_1} and {TICKER_2} ({PERIOD})...\n")

raw_data = yf.download(
    tickers=[TICKER_1, TICKER_2],
    period=PERIOD,
    interval='1d',
    auto_adjust=True
)

prices = raw_data['Close'][[TICKER_1, TICKER_2]].dropna()

print("DATA SUMMARY")
print("-" * 50)
print(f"  Period       : {prices.index[0].strftime('%Y-%m-%d')} to {prices.index[-1].strftime('%Y-%m-%d')}")
print(f"  Total Days   : {len(prices):,} trading days")
print(f"  {TICKER_1} Range    : ${prices[TICKER_1].min():.2f} - ${prices[TICKER_1].max():.2f}")
print(f"  {TICKER_2} Range   : ${prices[TICKER_2].min():.2f} - ${prices[TICKER_2].max():.2f}")
print("-" * 50)

prices.tail(10)

## Cell 3: Cointegration Test (Engle-Granger)
Uji apakah kedua aset memiliki hubungan cointegrated, yaitu spread bersifat mean-reverting.

In [ ]:
# ============================================================
# CELL 3: COINTEGRATION TEST (Engle-Granger Two-Step Method)
# ============================================================
# Kointegrasi berbeda dengan korelasi. Dua aset bisa berkorelasi
# tinggi tetapi tidak cointegrated. Kointegrasi berarti kombinasi
# linier dari kedua aset bersifat stasioner (mean-reverting).
#
# Hipotesis:
#   H0 : Kedua aset TIDAK cointegrated
#   H1 : Kedua aset cointegrated
#
# Keputusan:
#   p-value < 0.05  --> Tolak H0 --> Aset COINTEGRATED
#   p-value >= 0.05 --> Gagal Tolak H0 --> Tidak cukup bukti
# ------------------------------------------------------------

coint_t_stat, p_value, critical_values = coint(
    prices[TICKER_1],
    prices[TICKER_2]
)

correlation = prices[TICKER_1].corr(prices[TICKER_2])

print("COINTEGRATION TEST RESULTS (Engle-Granger)")
print("=" * 50)
print(f"  t-Statistic     : {coint_t_stat:.4f}")
print(f"  p-Value         : {p_value:.6f}")
print(f"  Critical Values : 1%: {critical_values[0]:.4f} | "
      f"5%: {critical_values[1]:.4f} | "
      f"10%: {critical_values[2]:.4f}")
print(f"  Pearson Corr.   : {correlation:.4f}")
print("-" * 50)

# Interpretasi hasil
if p_value < 0.05:
    print(f"  RESULT: {TICKER_1} dan {TICKER_2} TERBUKTI COINTEGRATED")
    print(f"  p-value ({p_value:.6f}) < 0.05")
    print(f"  Spread bersifat mean-reverting --> LAYAK untuk Pairs Trading")
else:
    print(f"  RESULT: Tidak cukup bukti bahwa {TICKER_1} dan {TICKER_2} cointegrated")
    print(f"  p-value ({p_value:.6f}) >= 0.05")
    print(f"  Pairs trading tetap bisa dilakukan, tetapi dengan risiko lebih tinggi")

print("=" * 50)

## Cell 4: Hedge Ratio & Spread Calculation
Menghitung hedge ratio menggunakan OLS regression, lalu membangun deret waktu spread.

In [ ]:
# ============================================================
# CELL 4: HEDGE RATIO & SPREAD via OLS REGRESSION
# ============================================================
# Hedge ratio menentukan berapa unit aset kedua (PEP) yang
# harus dipegang untuk setiap 1 unit aset pertama (KO) agar
# posisi menjadi market-neutral.
#
# Model OLS: KO = alpha + beta * PEP + epsilon
#   beta (slope)   = hedge ratio
#   epsilon (residu) = spread
# ------------------------------------------------------------

X = sm.add_constant(prices[TICKER_2])
Y = prices[TICKER_1]

ols_model   = sm.OLS(Y, X).fit()
alpha       = ols_model.params.iloc[0]
hedge_ratio = ols_model.params.iloc[1]
r_squared   = ols_model.rsquared

print("OLS REGRESSION RESULTS")
print("=" * 50)
print(f"  Model        : {TICKER_1} = alpha + beta * {TICKER_2}")
print(f"  Intercept    : {alpha:.4f}")
print(f"  Hedge Ratio  : {hedge_ratio:.4f}")
print(f"  R-Squared    : {r_squared:.4f} ({r_squared*100:.1f}%)")
print("-" * 50)
print(f"  Untuk setiap 1 lot {TICKER_1}, ambil posisi berlawanan")
print(f"  sebesar {abs(hedge_ratio):.4f} lot {TICKER_2}")
print("=" * 50)

# Hitung Spread = KO - (beta * PEP) - alpha
spread = prices[TICKER_1] - (hedge_ratio * prices[TICKER_2]) - alpha

print(f"\n  Spread Statistics:")
print(f"     Mean   : {spread.mean():.6f}")
print(f"     Std    : {spread.std():.4f}")
print(f"     Min    : {spread.min():.4f}")
print(f"     Max    : {spread.max():.4f}")

## Cell 5: Z-Score & Trading Signals
Menghitung rolling Z-score dan menentukan sinyal trading:
- **BUY Spread** ketika Z-Score < -2.0
- **SELL Spread** ketika Z-Score > +2.0

In [ ]:
# ============================================================
# CELL 5: ROLLING Z-SCORE & TRADING SIGNALS
# ============================================================
# Z = (Spread - Rolling_Mean) / Rolling_Std
#
# Trading Logic:
#   Z < -2.0 --> BUY spread  (Long KO, Short PEP)
#   Z > +2.0 --> SELL spread (Short KO, Long PEP)
# ------------------------------------------------------------

ZSCORE_WINDOW   = 30
ENTRY_THRESHOLD = 2.0

rolling_mean = spread.rolling(window=ZSCORE_WINDOW).mean()
rolling_std  = spread.rolling(window=ZSCORE_WINDOW).std()
z_score      = (spread - rolling_mean) / rolling_std

buy_signals  = z_score[z_score < -ENTRY_THRESHOLD]
sell_signals = z_score[z_score >  ENTRY_THRESHOLD]

current_z = z_score.iloc[-1]

print("Z-SCORE & TRADING SIGNALS")
print("=" * 50)
print(f"  Rolling Window  : {ZSCORE_WINDOW} days")
print(f"  Entry Threshold : +/-{ENTRY_THRESHOLD}")
print(f"  Current Z-Score : {current_z:.4f}")
print("-" * 50)
print(f"  BUY Signals     : {len(buy_signals)} occurrences")
print(f"  SELL Signals    : {len(sell_signals)} occurrences")
print("-" * 50)

if current_z < -ENTRY_THRESHOLD:
    print(f"  CURRENT SIGNAL: BUY SPREAD (Long {TICKER_1}, Short {TICKER_2})")
elif current_z > ENTRY_THRESHOLD:
    print(f"  CURRENT SIGNAL: SELL SPREAD (Short {TICKER_1}, Long {TICKER_2})")
else:
    print(f"  CURRENT SIGNAL: NEUTRAL -- No action required")

print("=" * 50)

## Cell 6: Interactive Visualization (Plotly)
Grafik interaktif dengan tema deep blue & black. Hover untuk melihat detail data.

In [ ]:
# ============================================================
# CELL 6: INTERACTIVE VISUALIZATION — Plotly (Deep Blue Theme)
# ============================================================
# Dua subplot interaktif:
#   1. Normalized Price Movement (KO vs PEP)
#   2. Z-Score dengan garis threshold +2, 0, -2
# 
# Fitur interaktif: hover, zoom, pan, download PNG
# ------------------------------------------------------------

# Normalize prices ke base 100
prices_norm = prices / prices.iloc[0] * 100

# Color palette
C_BG      = '#050A18'
C_PAPER   = '#000000'
C_GRID    = '#0D1B3E'
C_TEXT    = '#8899BB'
C_TITLE   = '#E8EDF5'
C_CYAN    = '#00E5FF'
C_MAGENTA = '#FF3D7F'
C_GOLD    = '#FFD700'
C_GREEN   = '#00FF88'
C_RED     = '#FF3D5A'
C_WHITE   = 'rgba(255,255,255,0.4)'

# Buat subplot figure
fig = make_subplots(
    rows=2, cols=1,
    row_heights=[0.55, 0.45],
    vertical_spacing=0.08,
    subplot_titles=[
        'NORMALIZED PRICE MOVEMENT (Base = 100)',
        f'Z-SCORE TRADING SIGNALS (Window = {ZSCORE_WINDOW}d, Threshold = +/-{ENTRY_THRESHOLD})'
    ]
)

# ---- SUBPLOT 1: Normalized Prices ----
fig.add_trace(
    go.Scatter(
        x=prices_norm.index, y=prices_norm[TICKER_1],
        name=f'{TICKER_1} (Coca-Cola)',
        line=dict(color=C_CYAN, width=1.8),
        hovertemplate='%{x|%Y-%m-%d}<br>' + TICKER_1 + ': %{y:.2f}<extra></extra>'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=prices_norm.index, y=prices_norm[TICKER_2],
        name=f'{TICKER_2} (PepsiCo)',
        line=dict(color=C_MAGENTA, width=1.8),
        hovertemplate='%{x|%Y-%m-%d}<br>' + TICKER_2 + ': %{y:.2f}<extra></extra>'
    ),
    row=1, col=1
)

# ---- SUBPLOT 2: Z-Score ----
fig.add_trace(
    go.Scatter(
        x=z_score.index, y=z_score,
        name='Z-Score',
        line=dict(color=C_CYAN, width=1.5),
        hovertemplate='%{x|%Y-%m-%d}<br>Z-Score: %{y:.3f}<extra></extra>'
    ),
    row=2, col=1
)

# Buy zones: area di bawah -2
z_buy = z_score.copy()
z_buy[z_buy >= -ENTRY_THRESHOLD] = -ENTRY_THRESHOLD
fig.add_trace(
    go.Scatter(
        x=z_score.index, y=z_buy,
        fill='tonexty', fillcolor='rgba(0,255,136,0.08)',
        line=dict(width=0), showlegend=False,
        hoverinfo='skip'
    ),
    row=2, col=1
)

# Sell zones: area di atas +2
z_sell = z_score.copy()
z_sell[z_sell <= ENTRY_THRESHOLD] = ENTRY_THRESHOLD
fig.add_trace(
    go.Scatter(
        x=z_score.index, y=z_sell,
        name='SELL Zone',
        fill='tonexty', fillcolor='rgba(255,61,90,0.08)',
        line=dict(width=0), showlegend=False,
        hoverinfo='skip'
    ),
    row=2, col=1
)

# Threshold lines: +2, 0, -2
for val, label, color, dash in [
    ( ENTRY_THRESHOLD, f'SELL (+{ENTRY_THRESHOLD})', C_RED,   'dash'),
    ( 0,               'MEAN (0)',                   C_WHITE, 'dash'),
    (-ENTRY_THRESHOLD, f'BUY (-{ENTRY_THRESHOLD})',  C_GREEN, 'dash'),
]:
    fig.add_hline(
        y=val, row=2, col=1,
        line=dict(color=color, width=1, dash=dash),
        annotation_text=label,
        annotation_position='right',
        annotation_font=dict(color=color, size=10, family='Courier New')
    )

# ---- LAYOUT: Deep Blue & Black Premium Theme ----
fig.update_layout(
    title=dict(
        text=(
            'Statistical Arbitrage: Pairs Trading Analysis<br>'
            '<span style="font-size:13px;color:#8899BB">'
            'Quantitative Research Portfolio by Rhameyza Faiqo Susanto</span>'
        ),
        font=dict(size=18, color=C_GOLD, family='Courier New'),
        x=0.5, xanchor='center'
    ),
    height=750,
    plot_bgcolor=C_BG,
    paper_bgcolor=C_PAPER,
    font=dict(color=C_TEXT, family='Courier New', size=11),
    legend=dict(
        bgcolor='rgba(10,15,32,0.9)',
        bordercolor=C_GRID,
        borderwidth=1,
        font=dict(color=C_TITLE, size=11)
    ),
    hovermode='x unified',
    margin=dict(t=100, b=60, l=60, r=30)
)

# Axis styling untuk kedua subplot
for i in [1, 2]:
    fig.update_xaxes(
        row=i, col=1,
        gridcolor=C_GRID, gridwidth=0.5,
        linecolor=C_GRID, zerolinecolor=C_GRID,
        tickfont=dict(color=C_TEXT, size=9)
    )
    fig.update_yaxes(
        row=i, col=1,
        gridcolor=C_GRID, gridwidth=0.5,
        linecolor=C_GRID, zerolinecolor=C_GRID,
        tickfont=dict(color=C_TEXT, size=9)
    )

# Subtitle styling
fig.update_annotations(
    font=dict(color=C_TITLE, size=12, family='Courier New')
)

# Annotation: statistik di subplot 1
fig.add_annotation(
    xref='x domain', yref='y domain',
    x=0.99, y=0.97,
    text=(
        f'Pearson r = {correlation:.4f}<br>'
        f'Coint. p  = {p_value:.4f}<br>'
        f'Hedge B   = {hedge_ratio:.4f}'
    ),
    showarrow=False,
    font=dict(color=C_GOLD, size=10, family='Courier New'),
    align='right',
    bgcolor='rgba(10,15,32,0.85)',
    bordercolor=C_GOLD,
    borderwidth=1,
    borderpad=6,
    row=1, col=1
)

fig.show()

## Cell 7: Executive Summary

In [ ]:
# ============================================================
# CELL 7: EXECUTIVE SUMMARY
# ============================================================

cointegrated_str = "YES" if p_value < 0.05 else "NO"

if current_z < -ENTRY_THRESHOLD:
    signal_str = f"BUY SPREAD (Long {TICKER_1}, Short {TICKER_2})"
elif current_z > ENTRY_THRESHOLD:
    signal_str = f"SELL SPREAD (Short {TICKER_1}, Long {TICKER_2})"
else:
    signal_str = "NEUTRAL -- No action"

print()
print("=" * 60)
print("  EXECUTIVE SUMMARY -- Pairs Trading Analysis")
print("  Quantitative Research by Rhameyza Faiqo Susanto")
print("=" * 60)
print(f"  Asset Pair     : {TICKER_1} (Coca-Cola) vs {TICKER_2} (PepsiCo)")
print(f"  Data Period    : {prices.index[0].strftime('%Y-%m-%d')} to {prices.index[-1].strftime('%Y-%m-%d')}")
print(f"  Sample Size    : {len(prices):,} trading days")
print("-" * 60)
print(f"  Pearson Corr.  : {correlation:.4f}")
print(f"  Coint. p-value : {p_value:.6f}")
print(f"  Cointegrated?  : {cointegrated_str}")
print("-" * 60)
print(f"  Hedge Ratio    : {hedge_ratio:.4f}")
print(f"  Intercept      : {alpha:.4f}")
print(f"  R-Squared      : {r_squared:.4f} ({r_squared*100:.1f}%)")
print("-" * 60)
print(f"  Z-Score Window : {ZSCORE_WINDOW} days")
print(f"  Current Z      : {current_z:.4f}")
print(f"  BUY Signals    : {len(buy_signals)} occurrences")
print(f"  SELL Signals   : {len(sell_signals)} occurrences")
print("-" * 60)
print(f"  CURRENT SIGNAL : {signal_str}")
print("=" * 60)
print()
print("  Disclaimer: For educational & research purposes only.")
print("  Not financial advice. Conduct your own due diligence.")